# ReAct 框架 - 第三部分：完整 Agent 实现

## 学习目标
1. 掌握 ReActAgent 的使用
2. 理解完整的执行流程
3. 学习错误处理和优化

## 目录
1. [ReActAgent 类](#1-reactagent-类)
2. [完整执行流程](#2-完整执行流程)
3. [错误处理](#3-错误处理)
4. [高级配置](#4-高级配置)
5. [实战案例](#5-实战案例)
6. [最佳实践](#6-最佳实践)

In [ ]:
import sys
sys.path.insert(0, '..')

from src.react import (
    SimpleTool, ReActAgent, ReActTrace,
    Thought, Action, Observation, ReActStep
)
print("模块加载成功！")

---
## 1. ReActAgent 类

### 1.1 创建工具集

In [ ]:
# 计算器工具
calculator = SimpleTool(
    name="calculator",
    description="执行数学计算",
    func=lambda x: str(eval(x))
)

# 搜索工具
def mock_search(query):
    data = {"python": "Python是编程语言", "北京": "北京是中国首都"}
    for k, v in data.items():
        if k in query.lower():
            return v
    return f"未找到: {query}"

search = SimpleTool(name="search", description="搜索信息", func=mock_search)

# 完成工具
finish = SimpleTool(
    name="finish",
    description="完成任务",
    func=lambda x: f"完成: {x}"
)

tools = [calculator, search, finish]
print(f"已创建 {len(tools)} 个工具")

### 1.2 创建 Agent

In [ ]:
# 创建 ReActAgent
agent = ReActAgent(tools=tools, max_iterations=5)

print(f"ReActAgent 配置：")
print(f"  工具数: {len(agent._tools)}")
print(f"  最大步数: {agent._max_iterations}")

### 1.3 运行 Agent

In [ ]:
# 运行任务（无LLM时返回模拟结果）
result = agent.run("计算 25 * 4 的结果")

print("执行结果：")
print(f"  任务: {result.question}")
print(f"  步骤数: {len(result.steps)}")
print(f"  完成: {result.success}")

---
## 2. 完整执行流程

### 2.1 流程图

In [ ]:
print("""
ReActAgent 执行流程：

┌─────────────────────────────────────────────────────┐
│  1. 接收任务                                         │
│     └─> 初始化 ReActTrace                           │
│                                                     │
│  2. 构建提示                                         │
│     └─> 包含工具描述和历史记录                        │
│                                                     │
│  3. 调用 LLM                                         │
│     └─> 获取 Thought + Action                       │
│                                                     │
│  4. 解析输出                                         │
│     └─> 提取工具名和输入                             │
│                                                     │
│  5. 执行工具                                         │
│     └─> 获取 Observation                            │
│                                                     │
│  6. 记录步骤                                         │
│     └─> 添加到 Trace                                │
│                                                     │
│  7. 检查终止条件                                     │
│     ├─> 任务完成 → 返回结果                          │
│     ├─> 达到最大步数 → 返回结果                      │
│     └─> 继续 → 回到步骤2                            │
└─────────────────────────────────────────────────────┘
""")

### 2.2 模拟完整流程

In [ ]:
# 手动模拟完整流程
print("模拟任务：计算2024年有多少小时")
print("="*50)

trace = ReActTrace(question="计算2024年有多少小时")

# 步骤1
step1 = ReActStep(
    thought=Thought(content="2024是闰年有366天，需要计算小时数"),
    action=Action(name="calculator", input="366 * 24"),
    observation=Observation(content="8784")
)
trace.add_step(step1)
print(f"\n步骤1:")
print(f"  Thought: {step1.thought.content}")
print(f"  Action: {step1.action.name}({step1.action.input})")
print(f"  Observation: {step1.observation.content}")

# 步骤2
step2 = ReActStep(
    thought=Thought(content="已得到结果，可以完成任务"),
    action=Action(name="finish", input="2024年有8784小时"),
    observation=Observation(content="任务完成")
)
trace.add_step(step2)
print(f"\n步骤2:")
print(f"  Thought: {step2.thought.content}")
print(f"  Action: {step2.action.name}({step2.action.input})")
print(f"  Observation: {step2.observation.content}")

print(f"\n总步骤数: {len(trace.steps)}")

---
## 3. 错误处理

### 3.1 常见错误类型

In [ ]:
print("""
ReAct 常见错误：

1. 工具不存在
   - LLM 调用了未注册的工具
   - 解决：返回错误提示，让 LLM 重试

2. 参数格式错误
   - 工具输入格式不正确
   - 解决：验证输入，提供格式说明

3. 工具执行失败
   - 工具内部错误
   - 解决：捕获异常，返回错误信息

4. 无限循环
   - LLM 重复相同操作
   - 解决：设置最大步数限制

5. 解析失败
   - LLM 输出格式不符合预期
   - 解决：使用容错解析器
""")

### 3.2 错误处理示例

In [ ]:
def safe_execute_tool(tool, input_str):
    """安全执行工具"""
    try:
        result = tool.execute(input_str)
        return {"success": True, "result": result}
    except Exception as e:
        return {"success": False, "error": str(e)}

# 测试正常执行
print("正常执行:")
print(safe_execute_tool(calculator, "10 + 5"))

# 测试错误执行
print("\n错误执行:")
print(safe_execute_tool(calculator, "invalid"))

---
## 4. 高级配置

### 4.1 配置选项

In [ ]:
print("""
ReActAgent 配置选项：

1. max_steps: 最大执行步数
   - 默认: 10
   - 建议: 5-15

2. tools: 可用工具列表
   - 必须包含 finish 工具

3. verbose: 是否输出详细日志
   - 调试时开启

4. timeout: 单步超时时间
   - 防止工具执行过长
""")

---
## 5. 实战案例

### 5.1 信息检索任务

In [ ]:
# 模拟信息检索
print("任务：查询Python的创建者")
print("="*40)

trace = ReActTrace(question="查询Python的创建者")

step = ReActStep(
    thought=Thought(content="需要搜索Python相关信息"),
    action=Action(name="search", input="Python创建者"),
    observation=Observation(content="Python由Guido van Rossum创建")
)
trace.add_step(step)

print(f"Thought: {step.thought.content}")
print(f"Action: search(Python创建者)")
print(f"Observation: {step.observation.content}")
print(f"\n答案: Python由Guido van Rossum创建")

### 5.2 多步骤计算

In [ ]:
# 多步骤计算示例
print("任务：计算圆的面积(半径=7, π=3.14)")
print("="*40)

print("\n步骤1:")
print("  Thought: 圆面积公式是 π*r²，先计算 r²")
print("  Action: calculator(7 * 7)")
print("  Observation: 49")

print("\n步骤2:")
print("  Thought: 现在计算 π*49")
print("  Action: calculator(3.14 * 49)")
print("  Observation: 153.86")

print("\n步骤3:")
print("  Thought: 已得到结果")
print("  Action: finish(圆的面积是153.86平方单位)")
print("  Observation: 任务完成")

---
## 6. 最佳实践

In [ ]:
print("""
ReAct 最佳实践：

1. 工具设计
   - 功能单一，描述清晰
   - 输入输出格式明确
   - 包含错误处理

2. 提示工程
   - 提供清晰的工具说明
   - 包含使用示例
   - 明确输出格式要求

3. 执行控制
   - 设置合理的最大步数
   - 实现超时机制
   - 记录执行日志

4. 错误恢复
   - 捕获并处理异常
   - 提供有用的错误信息
   - 允许重试机制
""")

---
## 总结

ReAct 系列教程覆盖：
- **Part 1**: 基础概念和数据结构
- **Part 2**: 工具定义和提示构建
- **Part 3**: 完整 Agent 实现

下一步学习 **03_TreeOfThoughts_tutorial.ipynb**